# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.### Dataset SourceThe dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint
import json

# Define the dataset Croissant JSON-LD URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset name:", metadata.name)
print("Description:", metadata.description)
print("Cite as:", getattr(metadata, 'cite_as', getattr(metadata, 'citeAs', '')))  # citeAs for compatibility
print("Date published:", getattr(metadata, 'datePublished', getattr(metadata, 'date_published', '')))
print("Keywords:", getattr(metadata, 'keywords', ''))

## 2. Data Overview
Review available record sets, fields, and their IDs. All references use their `@id` for consistency.

In [ ]:
# List available record sets by @id

record_sets = []
if hasattr(metadata, 'record_sets') and metadata.record_sets:
    # Modern mlcroissant v0.8+
    record_sets = metadata.record_sets
elif hasattr(metadata, 'recordSet') and metadata.recordSet:
    # Legacy key
    record_sets = metadata.recordSet

if not record_sets:
    # If record sets are missing from metadata, get from dataset directly
    try:
        record_sets = dataset.record_sets
    except Exception:
        print("No record sets found.")

if not record_sets:
    print("This dataset may not explicitly declare record sets. Attempting to enumerate data...")
    # Try with the dataset API
    # Many Croissant datasets have a single record set with @id 'cr:ExperimentDataset-Records', but not always
    # Print basic structure to help user navigate
    pprint.pprint(vars(metadata))
else:
    print(f"Found {len(record_sets)} record set(s):")
    for rs in record_sets:
        if hasattr(rs, '@id'):
            print(f"- Record Set @id: {rs['@id'] if isinstance(rs, dict) else rs.@id}")
        elif isinstance(rs, str):
            print(f"- Record Set @id: {rs}")

In [ ]:
# For each record set, print available fields by @id
print("\nField IDs for each record set:")

def _get_field_list(record_set):
    """
    Given a record set object, returns a list of field dicts (with @id etc).
    """
    fs = []
    if isinstance(record_set, dict):
        fs = record_set.get('field', []) or record_set.get('fields', [])
    elif hasattr(record_set, 'field'):
        fs = record_set.field
    elif hasattr(record_set, 'fields'):
        fs = record_set.fields
    return fs

if record_sets:
    for i, record_set in enumerate(record_sets):
        if hasattr(record_set, '@id'):
            recset_id = record_set['@id'] if isinstance(record_set, dict) else record_set.@id
        elif isinstance(record_set, str):
            recset_id = record_set
        else:
            recset_id = str(i)
        print(f"\nRecord Set @id: {recset_id}")
        # Retrieve field list
        fields = _get_field_list(record_set)
        if fields:
            for f in fields:
                if isinstance(f, dict):
                    print(f"- Field @id: {f.get('@id', '')}\tName: {f.get('name', '')}")
                elif hasattr(f, '@id'):
                    print(f"- Field @id: {f.@id}\tName: {getattr(f, 'name', '')}")
                else:
                    print(f"- Field: {f}")
        else:
            print("(No fields declared in this record set)")
else:
    print("(No record sets found)")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All entities are referenced by `@id`.

In [ ]:
# ---
# Set up: If no record_set is found above, let's try main plausible @id.
# Replace these variables as needed with @id values from above overview.

# Default (if provided above, else update for your dataset):
RECORD_SET_IDS = []

# If record_sets were found, collect their @id
if record_sets:
    for rs in record_sets:
        if hasattr(rs, '@id'):
            recset_id = rs['@id'] if isinstance(rs, dict) else rs.@id
        elif isinstance(rs, str):
            recset_id = rs
        else:
            recset_id = None
        if recset_id:
            RECORD_SET_IDS.append(recset_id)

if not RECORD_SET_IDS:
    # Guess: often main records are in cr:ExperimentDataset or cr:RecordSet with '-Records', if not, try default
    RECORD_SET_IDS = ["cr:ExperimentDataset-Records", "cr:RecordSet-1"]

# ---
# Load records for each record set into pandas DataFrames
dataframes = {}
for recset_id in RECORD_SET_IDS:
    print(f"Attempting to load records for record set @id: {recset_id}")
    try:
        records = list(dataset.records(record_set=recset_id))
        if records:
            dataframes[recset_id] = pd.DataFrame(records)
    except Exception as e:
        print(f"  Could not load records for {recset_id}: {e}")

# Report results and show first few columns
if dataframes:
    recset0 = list(dataframes.keys())[0]
    print(f"Fields for record set {recset0}:\n", dataframes[recset0].columns.tolist())
    display(dataframes[recset0].head())
else:
    print("No records could be extracted. Please check available record set @ids.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on a numeric field, normalizing, and grouping. All references by `@id`.

In [ ]:
# Choose a numeric field and a categorical/group field based on what is in the data

DEFAULT_NUMERIC_FIELDS = [
    "cr:coverage_percentage",
    "cr:peptide_count",
    "cr:MW",
    "cr:pI",
    "cr:normalized_abundance_sample_1",
    "cr:normalized_abundance_sample_2",
    "cr:normalized_abundance_sample_3",
]
DEFAULT_GROUP_FIELDS = [
    "cr:description",
    "cr:protein_accession",
    "cr:modification_type"
]

main_recset_id = list(dataframes.keys())[0] if dataframes else None
df = dataframes[main_recset_id] if main_recset_id else None

numeric_field = None
for f in DEFAULT_NUMERIC_FIELDS:
    if df is not None and f in df.columns:
        numeric_field = f
        break
if not numeric_field:
    # Fallback: First float/integer column
    possible_numeric = df.select_dtypes(include=['float', 'int']).columns.tolist() if df is not None else []
    numeric_field = possible_numeric[0] if possible_numeric else None

group_field = None
for f in DEFAULT_GROUP_FIELDS:
    if df is not None and f in df.columns:
        group_field = f
        break
if not group_field:
    # Fallback: any object/categorical
    possible_groups = df.select_dtypes(include=['object']).columns.tolist() if df is not None else []
    group_field = possible_groups[0] if possible_groups else None

print(f"Selected numeric field: {numeric_field}")
print(f"Selected group field: {group_field}")

if numeric_field and group_field:
    # Safe: Cast column to numeric
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
    print(f"Applying threshold for {numeric_field} > {threshold:.2f}")
    filtered_df = df[df[numeric_field] > threshold].copy()

    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Standard score normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by group_field and mean
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"\nGrouped mean values by {group_field}:")
        display(grouped_df.head())
else:
    print("Not enough fields to proceed with numeric EDA. Please adjust field ids.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Example: Histogram and boxplot of the main numeric field, colored/stratified by group if available
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Histogram of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    if group_field:
        # Boxplot
        plt.figure(figsize=(12,5))
        # For speed, only show up to 20 categories
        top_groups = df[group_field].value_counts().nlargest(20).index
        sns.boxplot(x=group_field, y=numeric_field, data=df[df[group_field].isin(top_groups)])
        plt.title(f"{numeric_field} by {group_field} (top 20 groups)")
        plt.xticks(rotation=60)
        plt.show()
else:
    print("Not enough data to plot.")

## 6. Conclusion
This notebook demonstrated the use of the `mlcroissant` library to load protein abundance and modification data from the Croissant FAIR^2 dataset package. We reviewed the structure (record sets and fields by `@id`), performed basic filtering, normalization, grouping, and visualization on numeric fields. You can extend this notebook by referencing additional record sets or fields, or by applying domain-specific analyses for proteomics.